In [10]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical



In [14]:
# Download NLTK stopwords
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rheah\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [19]:
df=pd.read_csv('november_chunk_1.csv')

In [25]:
# Convert 'epoch' column to datetime
df['date'] = pd.to_datetime(df['epoch'], unit='s')

In [28]:

# Drop the old 'epoch' column if not needed
df.drop(columns=['epoch'], inplace=True)

In [29]:
df.head()

,0,type,id,username,text,url,media,retweetedTweet,retweetedTweetID,retweetedUserID,...,links,viewCount,quotedTweet,in_reply_to_screen_name,in_reply_to_status_id_str,in_reply_to_user_id_str,location,cash_app_handle,user,date
0,NaN,tweet-,1.859386e+18,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,https://twitter.com/SATraveler2/status/1859386...,[],False,NaN,NaN,...,[{'display_url': 'childrenshealthdefense.org/d...,"{'count': '4', 'state': 'EnabledWithCount'}",False,MdBreathe,1.859234e+18,1.139217e+18,NaN,NaN,"{'id': 1280313070053502978, 'id_str': '1280313...",2024-11-20 23:59:59
1,NaN,tweet-,1.859386e+18,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,https://twitter.com/JoeZumwalt/status/18593862...,[],False,NaN,NaN,...,[],"{'count': '3', 'state': 'EnabledWithCount'}",False,UROCKlive1,1.859057e+18,8.798304e+07,NaN,NaN,"{'id': 1087502843102162945, 'id_str': '1087502...",2024-11-20 23:59:57
2,NaN,tweet-,1.859386e+18,67sharona,Laken Rileys Death Would Of Never Ever Happene...,https://twitter.com/67sharona/status/185938622...,[],False,NaN,NaN,...,[],"{'count': '25', 'state': 'EnabledWithCount'}",False,NaN,NaN,NaN,NaN,NaN,"{'id': 792933462256652288, 'id_str': '79293346...",2024-11-20 23:59:56
3,NaN,tweet-,1.859386e+18,TruthAboutF,POV: you go downstairs in the middle of the ni...,https://twitter.com/TruthAboutF/status/1859386...,"[{'display_url': 'pic.x.com/tUK0JjrAHY', 'expa...",False,NaN,NaN,...,[],"{'count': '1476', 'state': 'EnabledWithCount'}",False,NaN,NaN,NaN,NaN,NaN,"{'id': 1424726838144901122, 'id_str': '1424726...",2024-11-20 23:59:55
4,NaN,tweet-,1.859386e+18,Hump98Clint,"@mtgreenee This is a good question, are you no...",https://twitter.com/Hump98Clint/status/1859386...,[],False,NaN,NaN,...,[],"{'count': '8', 'state': 'EnabledWithCount'}",False,mtgreenee,1.859042e+18,8.260652e+17,NaN,NaN,"{'id': 1942063262, 'id_str': '1942063262', 'ur...",2024-11-20 23:59:53


In [30]:
# Drop unnecessary columns
columns_to_drop = [
    'type', 'id', 'url', 'media', 'retweetedTweet', 'retweetedTweetID', 
    'links', 'viewCount', 'quotedTweet', 'in_reply_to_status_id_str', 
    'in_reply_to_user_id_str', 'cash_app_handle'
]

df.drop(columns=columns_to_drop, inplace=True, errors='ignore')


In [31]:
df.head()

,0,username,text,retweetedUserID,id_str,lang,rawContent,replyCount,retweetCount,likeCount,quoteCount,conversationId,conversationIdStr,hashtags,mentionedUsers,in_reply_to_screen_name,location,user,date
0,NaN,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,NaN,1.859386e+18,en,@MdBreathe continued- \nto develop a techniqu...,0.0,0.0,0.0,0.0,1.859234e+18,1.859234e+18,[],"[{'id_str': '1139217161275101184', 'name': 'Ma...",MdBreathe,NaN,"{'id': 1280313070053502978, 'id_str': '1280313...",2024-11-20 23:59:59
1,NaN,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,NaN,1.859386e+18,en,@UROCKlive1 all these banderite/nazi sycophant...,0.0,0.0,0.0,0.0,1.859057e+18,1.859057e+18,[],"[{'id_str': '87983041', 'name': 'Khashoggi’s G...",UROCKlive1,NaN,"{'id': 1087502843102162945, 'id_str': '1087502...",2024-11-20 23:59:57
2,NaN,67sharona,Laken Rileys Death Would Of Never Ever Happene...,NaN,1.859386e+18,en,Laken Rileys Death Would Of Never Ever Happene...,0.0,0.0,1.0,0.0,1.859386e+18,1.859386e+18,"[{'indices': [98, 119], 'text': 'justiceforlak...","[{'id_str': '25073877', 'name': 'Donald J. Tru...",NaN,NaN,"{'id': 792933462256652288, 'id_str': '79293346...",2024-11-20 23:59:56
3,NaN,TruthAboutF,POV: you go downstairs in the middle of the ni...,NaN,1.859386e+18,en,POV: you go downstairs in the middle of the ni...,0.0,2.0,14.0,0.0,1.859386e+18,1.859386e+18,[],[],NaN,NaN,"{'id': 1424726838144901122, 'id_str': '1424726...",2024-11-20 23:59:55
4,NaN,Hump98Clint,"@mtgreenee This is a good question, are you no...",NaN,1.859386e+18,en,"@mtgreenee This is a good question, are you no...",0.0,0.0,1.0,0.0,1.859042e+18,1.859042e+18,[],"[{'id_str': '826065164504006657', 'name': 'Mar...",mtgreenee,NaN,"{'id': 1942063262, 'id_str': '1942063262', 'ur...",2024-11-20 23:59:53


In [35]:
# Drop unnecessary columns
Columns_to_drop = ['0',  'retweetedUserID', 'id_str', 'lang',
       'rawContent', 'replyCount', 'retweetCount',  
       'conversationId', 'conversationIdStr', 'hashtags', 'mentionedUsers',
       'in_reply_to_screen_name', 'location', 'user', ]

df.drop(columns=Columns_to_drop, inplace=True, errors='ignore')

In [38]:
df.head(10)

,username,text,likeCount,quoteCount,date
0,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,0.0,0.0,2024-11-20 23:59:59
1,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,0.0,0.0,2024-11-20 23:59:57
2,67sharona,Laken Rileys Death Would Of Never Ever Happene...,1.0,0.0,2024-11-20 23:59:56
3,TruthAboutF,POV: you go downstairs in the middle of the ni...,14.0,0.0,2024-11-20 23:59:55
4,Hump98Clint,"@mtgreenee This is a good question, are you no...",1.0,0.0,2024-11-20 23:59:53
5,BlakeBurman,The race is on for the next DNC chair. Martin ...,3.0,1.0,2024-11-20 23:59:51
6,Chip_quicksteps,@DefiyantlyFree Kamala Harris is a leech grift...,0.0,0.0,2024-11-20 23:59:50
7,Cynthia19941119,@BarronTrumpoo Kamala Harris is Vice President...,0.0,0.0,2024-11-20 23:59:49
8,G2638Garr,@MayorOfLA @POTUS A Pew study this year fall s...,0.0,0.0,2024-11-20 23:59:49
9,mascotmike12,@TheDemocrats Wtf are you doing Montana?,7.0,0.0,2024-11-20 23:59:45


In [39]:
df.shape

(50000, 5)

In [40]:
# Select necessary columns
df_new = df[['username', 'text', 'likeCount', 'quoteCount', 'date']]

# Save to CSV without the index
df_new.to_csv('election.csv', index=False)

print("✅ New dataset saved as 'election.csv' successfully!")



✅ New dataset saved as 'election.csv' successfully!


In [41]:
!pip install pandas nltk


# Preprocessing for Sentiment Analysis

In [43]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer


In [44]:

# Download VADER lexicon
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Rheah\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [45]:
# Load dataset
df = pd.read_csv('election.csv')

In [46]:
# Initialize VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

In [47]:
# Function for text cleaning (removes URLs, mentions, hashtags, special characters)
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r"http\S+|www\S+", "", text)  # Remove URLs
    text = re.sub(r"@\w+|\#", "", text)  # Remove mentions and hashtags
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # Remove special characters
    return text

In [48]:
# Apply text cleaning
df['clean_text'] = df['text'].astype(str).apply(clean_text)

In [49]:
df.head()

,username,text,likeCount,quoteCount,date,clean_text
0,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,0.0,0.0,2024-11-20 23:59:59,continued \nto develop a technique for hidin...
1,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,0.0,0.0,2024-11-20 23:59:57,all these banderitenazi sycophants to the dnc
2,67sharona,Laken Rileys Death Would Of Never Ever Happene...,1.0,0.0,2024-11-20 23:59:56,laken rileys death would of never ever happene...
3,TruthAboutF,POV: you go downstairs in the middle of the ni...,14.0,0.0,2024-11-20 23:59:55,pov you go downstairs in the middle of the nig...
4,Hump98Clint,"@mtgreenee This is a good question, are you no...",1.0,0.0,2024-11-20 23:59:53,this is a good question are you not a member ...


In [50]:
# Initialize stopwords, lemmatizer, and VADER sentiment analyzer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [53]:
# Function for tokenization, stopword removal, and lemmatization
def preprocess_text(text):
    tokens = text.split()  # Simple split instead of word_tokenize()
    tokens = [word for word in tokens if word.lower() not in stop_words]  # Remove stopwords
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
    return " ".join(tokens)  # Convert list back to string


In [54]:
# Apply tokenization, stopword removal, and lemmatization
df['processed_text'] = df['clean_text'].apply(preprocess_text)

In [55]:
df.head()

,username,text,likeCount,quoteCount,date,clean_text,processed_text
0,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,0.0,0.0,2024-11-20 23:59:59,continued \nto develop a technique for hidin...,continued develop technique hiding evidence hu...
1,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,0.0,0.0,2024-11-20 23:59:57,all these banderitenazi sycophants to the dnc,banderitenazi sycophant dnc
2,67sharona,Laken Rileys Death Would Of Never Ever Happene...,1.0,0.0,2024-11-20 23:59:56,laken rileys death would of never ever happene...,laken riley death would never ever happened pr...
3,TruthAboutF,POV: you go downstairs in the middle of the ni...,14.0,0.0,2024-11-20 23:59:55,pov you go downstairs in the middle of the nig...,pov go downstairs middle night bowl froot loop...
4,Hump98Clint,"@mtgreenee This is a good question, are you no...",1.0,0.0,2024-11-20 23:59:53,this is a good question are you not a member ...,good question member republican party list peo...


# Using Spacy

In [56]:
import spacy


In [57]:
# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

In [58]:
# Load dataset
df = pd.read_csv("election.csv")

In [59]:
# Function to clean and preprocess text
def preprocess_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)  # Remove URLs
    text = re.sub(r"@\w+|\#", '', text)  # Remove mentions and hashtags
    text = re.sub(r"[^\w\s]", '', text)  # Remove punctuation
    text = text.lower().strip()  # Convert to lowercase

    # Process text using spaCy
    doc = nlp(text)
    
    # Tokenization, stopword removal, and lemmatization
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]

    return " ".join(tokens)  # Convert back to string

In [60]:
# Apply preprocessing to the text column
df["processed_text"] = df["text"].astype(str).apply(preprocess_text)

In [61]:
df.head()

,username,text,likeCount,quoteCount,date,processed_text
0,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,0.0,0.0,2024-11-20 23:59:59,continue \n develop technique hide evidence h...
1,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,0.0,0.0,2024-11-20 23:59:57,banderitenazi sycophant dnc
2,67sharona,Laken Rileys Death Would Of Never Ever Happene...,1.0,0.0,2024-11-20 23:59:56,laken rileys death happen president justic...
3,TruthAboutF,POV: you go downstairs in the middle of the ni...,14.0,0.0,2024-11-20 23:59:55,pov downstairs middle night bowl froot loop ft...
4,Hump98Clint,"@mtgreenee This is a good question, are you no...",1.0,0.0,2024-11-20 23:59:53,good question member republican party list people


In [62]:
text = "Hello! This is a sample tweet. #NLP @user123"
doc = nlp(text)

tokens = [token.text for token in doc]
print(tokens)

['Hello', '!', 'This', 'is', 'a', 'sample', 'tweet', '.', '#', 'NLP', '@user123']


In [64]:
# Tokenization
df["Tokens"] = df["processed_text"].apply(lambda x: [token.text for token in nlp(str(x))])

In [65]:
# Stopword Removal
df["Filtered_Tokens"] = df["processed_text"].apply(lambda x: [token.text for token in nlp(str(x)) if not token.is_stop])

In [66]:
# Lemmatization
df["Lemmatized_Tokens"] = df["processed_text"].apply(lambda x: [token.lemma_ for token in nlp(str(x))])

In [68]:
# POS Tagging
df["POS_Tags"] = df["processed_text"].apply(lambda x: [(token.text, token.pos_) for token in nlp(str(x))])

In [69]:
# Display processed dataframe
print(df[["processed_text", "Tokens", "Filtered_Tokens", "Lemmatized_Tokens", "POS_Tags"]].head())

                                      processed_text  \
0  continue  \n develop technique hide evidence h...   
1                        banderitenazi sycophant dnc   
2  laken rileys death happen   president   justic...   
3  pov downstairs middle night bowl froot loop ft...   
4  good question member republican party list people   

                                              Tokens  \
0  [continue,  \n , develop, technique, hide, evi...   
1                    [banderitenazi, sycophant, dnc]   
2  [laken, rileys, death, happen,   , president, ...   
3  [pov, downstairs, middle, night, bowl, froot, ...   
4  [good, question, member, republican, party, li...   

                                     Filtered_Tokens  \
0  [continue,  \n , develop, technique, hide, evi...   
1                    [banderitenazi, sycophant, dnc]   
2  [laken, rileys, death, happen,   , president, ...   
3  [pov, downstairs, middle, night, bowl, froot, ...   
4  [good, question, member, republican, party,

In [70]:
df.head()

,username,text,likeCount,quoteCount,date,processed_text,Tokens,Filtered_Tokens,Lemmatized_Tokens,POS_Tags
0,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,0.0,0.0,2024-11-20 23:59:59,continue \n develop technique hide evidence h...,"[continue, \n , develop, technique, hide, evi...","[continue, \n , develop, technique, hide, evi...","[continue, \n , develop, technique, hide, evi...","[(continue, VERB), ( \n , SPACE), (develop, VE..."
1,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,0.0,0.0,2024-11-20 23:59:57,banderitenazi sycophant dnc,"[banderitenazi, sycophant, dnc]","[banderitenazi, sycophant, dnc]","[banderitenazi, sycophant, dnc]","[(banderitenazi, PROPN), (sycophant, PROPN), (..."
2,67sharona,Laken Rileys Death Would Of Never Ever Happene...,1.0,0.0,2024-11-20 23:59:56,laken rileys death happen president justic...,"[laken, rileys, death, happen, , president, ...","[laken, rileys, death, happen, , president, ...","[laken, rileys, death, happen, , president, ...","[(laken, VERB), (rileys, PROPN), (death, NOUN)..."
3,TruthAboutF,POV: you go downstairs in the middle of the ni...,14.0,0.0,2024-11-20 23:59:55,pov downstairs middle night bowl froot loop ft...,"[pov, downstairs, middle, night, bowl, froot, ...","[pov, downstairs, middle, night, bowl, froot, ...","[pov, downstairs, middle, night, bowl, froot, ...","[(pov, PROPN), (downstairs, ADV), (middle, ADJ..."
4,Hump98Clint,"@mtgreenee This is a good question, are you no...",1.0,0.0,2024-11-20 23:59:53,good question member republican party list people,"[good, question, member, republican, party, li...","[good, question, member, republican, party, li...","[good, question, member, republican, party, li...","[(good, ADJ), (question, NOUN), (member, NOUN)..."


In [71]:
# Function to clean text
def clean_text(text):
    if isinstance(text, str):  # Ensure it's a string
        text = text.replace("\n", " ")  # Remove newlines
        text = " ".join(text.split())  # Remove extra spaces
    return text

# Apply cleaning to relevant text columns
text_columns = ["processed_text", "Tokens", "Filtered_Tokens", "Lemmatized_Tokens", "POS_Tags"]
for col in text_columns:
    df[col] = df[col].astype(str).apply(clean_text)

In [72]:
df.head()

,username,text,likeCount,quoteCount,date,processed_text,Tokens,Filtered_Tokens,Lemmatized_Tokens,POS_Tags
0,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,0.0,0.0,2024-11-20 23:59:59,continue develop technique hide evidence human...,"['continue', ' \n ', 'develop', 'technique', '...","['continue', ' \n ', 'develop', 'technique', '...","['continue', ' \n ', 'develop', 'technique', '...","[('continue', 'VERB'), (' \n ', 'SPACE'), ('de..."
1,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,0.0,0.0,2024-11-20 23:59:57,banderitenazi sycophant dnc,"['banderitenazi', 'sycophant', 'dnc']","['banderitenazi', 'sycophant', 'dnc']","['banderitenazi', 'sycophant', 'dnc']","[('banderitenazi', 'PROPN'), ('sycophant', 'PR..."
2,67sharona,Laken Rileys Death Would Of Never Ever Happene...,1.0,0.0,2024-11-20 23:59:56,laken rileys death happen president justicefor...,"['laken', 'rileys', 'death', 'happen', ' ', 'p...","['laken', 'rileys', 'death', 'happen', ' ', 'p...","['laken', 'rileys', 'death', 'happen', ' ', 'p...","[('laken', 'VERB'), ('rileys', 'PROPN'), ('dea..."
3,TruthAboutF,POV: you go downstairs in the middle of the ni...,14.0,0.0,2024-11-20 23:59:55,pov downstairs middle night bowl froot loop ft...,"['pov', 'downstairs', 'middle', 'night', 'bowl...","['pov', 'downstairs', 'middle', 'night', 'bowl...","['pov', 'downstairs', 'middle', 'night', 'bowl...","[('pov', 'PROPN'), ('downstairs', 'ADV'), ('mi..."
4,Hump98Clint,"@mtgreenee This is a good question, are you no...",1.0,0.0,2024-11-20 23:59:53,good question member republican party list people,"['good', 'question', 'member', 'republican', '...","['good', 'question', 'member', 'republican', '...","['good', 'question', 'member', 'republican', '...","[('good', 'ADJ'), ('question', 'NOUN'), ('memb..."


In [73]:
import ast  # For converting string representation of lists to actual lists


In [74]:
# Function to clean text inside lists
def clean_list_column(text):
    if isinstance(text, str):
        try:
            # Convert string to list
            tokens = ast.literal_eval(text)  
            # Remove "\n" and extra spaces
            cleaned_tokens = [token.strip().replace("\n", "") for token in tokens if token.strip()]
            return cleaned_tokens
        except:
            return text  # If conversion fails, return original text
    return text


In [75]:
# Apply cleaning to list-like columns
list_columns = ["Tokens", "Filtered_Tokens", "Lemmatized_Tokens", "POS_Tags"]
for col in list_columns:
    df[col] = df[col].apply(clean_list_column)

In [76]:
# Clean regular text columns
df["processed_text"] = df["processed_text"].str.replace("\n", " ").str.strip()

In [77]:
df.head()

,username,text,likeCount,quoteCount,date,processed_text,Tokens,Filtered_Tokens,Lemmatized_Tokens,POS_Tags
0,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,0.0,0.0,2024-11-20 23:59:59,continue develop technique hide evidence human...,"[continue, develop, technique, hide, evidence,...","[continue, develop, technique, hide, evidence,...","[continue, develop, technique, hide, evidence,...","[('continue', 'VERB'), (' \n ', 'SPACE'), ('de..."
1,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,0.0,0.0,2024-11-20 23:59:57,banderitenazi sycophant dnc,"[banderitenazi, sycophant, dnc]","[banderitenazi, sycophant, dnc]","[banderitenazi, sycophant, dnc]","[('banderitenazi', 'PROPN'), ('sycophant', 'PR..."
2,67sharona,Laken Rileys Death Would Of Never Ever Happene...,1.0,0.0,2024-11-20 23:59:56,laken rileys death happen president justicefor...,"[laken, rileys, death, happen, president, just...","[laken, rileys, death, happen, president, just...","[laken, rileys, death, happen, president, just...","[('laken', 'VERB'), ('rileys', 'PROPN'), ('dea..."
3,TruthAboutF,POV: you go downstairs in the middle of the ni...,14.0,0.0,2024-11-20 23:59:55,pov downstairs middle night bowl froot loop ft...,"[pov, downstairs, middle, night, bowl, froot, ...","[pov, downstairs, middle, night, bowl, froot, ...","[pov, downstairs, middle, night, bowl, froot, ...","[('pov', 'PROPN'), ('downstairs', 'ADV'), ('mi..."
4,Hump98Clint,"@mtgreenee This is a good question, are you no...",1.0,0.0,2024-11-20 23:59:53,good question member republican party list people,"[good, question, member, republican, party, li...","[good, question, member, republican, party, li...","[good, question, member, republican, party, li...","[('good', 'ADJ'), ('question', 'NOUN'), ('memb..."


In [78]:
# Function to clean tweets
def clean_tweet(text):
    if isinstance(text, str):
        text = text.lower()  # Convert to lowercase
        text = re.sub(r'@\w+', '', text)  # Remove mentions (@usernames)
        text = re.sub(r'http\S+', '', text)  # Remove URLs
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
        text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
        return text
    return text


In [79]:
# Function to clean list columns (Tokens, Filtered_Tokens, etc.)
def clean_list_column(text):
    if isinstance(text, str):
        try:
            tokens = ast.literal_eval(text)  
            cleaned_tokens = [token.strip().replace("\n", "") for token in tokens if token.strip()]
            return cleaned_tokens
        except:
            return text
    return text

# Apply cleaning to relevant columns
df["cleaned_tweet"] = df["text"].apply(clean_tweet)  # Create cleaned tweet column
list_columns = ["Tokens", "Filtered_Tokens", "Lemmatized_Tokens", "POS_Tags"]
for col in list_columns:
    df[col] = df[col].apply(clean_list_column)

In [80]:
# Clean processed_text column
df["processed_text"] = df["processed_text"].str.replace("\n", " ").str.strip()

In [81]:
df.head()

,username,text,likeCount,quoteCount,date,processed_text,Tokens,Filtered_Tokens,Lemmatized_Tokens,POS_Tags,cleaned_tweet
0,SATraveler2,@MdBreathe continued- \nto develop a techniqu...,0.0,0.0,2024-11-20 23:59:59,continue develop technique hide evidence human...,"[continue, develop, technique, hide, evidence,...","[continue, develop, technique, hide, evidence,...","[continue, develop, technique, hide, evidence,...","[('continue', 'VERB'), (' \n ', 'SPACE'), ('de...",continued to develop a technique for hiding ev...
1,JoeZumwalt,@UROCKlive1 all these banderite/nazi sycophant...,0.0,0.0,2024-11-20 23:59:57,banderitenazi sycophant dnc,"[banderitenazi, sycophant, dnc]","[banderitenazi, sycophant, dnc]","[banderitenazi, sycophant, dnc]","[('banderitenazi', 'PROPN'), ('sycophant', 'PR...",all these banderitenazi sycophants to the dnc
2,67sharona,Laken Rileys Death Would Of Never Ever Happene...,1.0,0.0,2024-11-20 23:59:56,laken rileys death happen president justicefor...,"[laken, rileys, death, happen, president, just...","[laken, rileys, death, happen, president, just...","[laken, rileys, death, happen, president, just...","[('laken', 'VERB'), ('rileys', 'PROPN'), ('dea...",laken rileys death would of never ever happene...
3,TruthAboutF,POV: you go downstairs in the middle of the ni...,14.0,0.0,2024-11-20 23:59:55,pov downstairs middle night bowl froot loop ft...,"[pov, downstairs, middle, night, bowl, froot, ...","[pov, downstairs, middle, night, bowl, froot, ...","[pov, downstairs, middle, night, bowl, froot, ...","[('pov', 'PROPN'), ('downstairs', 'ADV'), ('mi...",pov you go downstairs in the middle of the nig...
4,Hump98Clint,"@mtgreenee This is a good question, are you no...",1.0,0.0,2024-11-20 23:59:53,good question member republican party list people,"[good, question, member, republican, party, li...","[good, question, member, republican, party, li...","[good, question, member, republican, party, li...","[('good', 'ADJ'), ('question', 'NOUN'), ('memb...",this is a good question are you not a member o...


In [85]:
print(df.isnull().sum())


username             0
text                 0
likeCount            0
quoteCount           0
date                 0
processed_text       0
Tokens               0
Filtered_Tokens      0
Lemmatized_Tokens    0
POS_Tags             0
cleaned_tweet        0
dtype: int64


In [82]:
# Save cleaned dataset
df.to_csv("election_cleaned.csv", index=False)
print("Dataset cleaned successfully with 'cleaned_tweet' column")

Dataset cleaned successfully with 'cleaned_tweet' column
